# Task 3: A/B Hypothesis Testing

## Objective
Statistically validate or reject key hypotheses about risk drivers, forming the evidence base for ACIS's new segmentation and pricing strategy.

**KPIs defined:**
- **Claim Frequency**: Proportion of policies with at least one claim.
- **Claim Severity**: Average claim amount, given a claim occurred.
- **Margin**: TotalPremium − TotalClaims.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath('../src'))

from data_loader import load_raw_data, preprocess
from hypothesis_tests import (
    test_province_risk, test_zipcode_risk, 
    test_margin_zipcode, test_gender_risk, 
    build_results_table
)

pd.set_option('display.max_columns', None)
print("Environment ready.")

Environment ready.


## 1. Load and Preprocess Data

In [2]:
DATA_PATH = r'c:\KAIM\MachineLearningRating_v3.txt'

try:
    df_raw = load_raw_data(DATA_PATH)
    df = preprocess(df_raw)
    print(f"Data loaded successfully: {df.shape[0]:,} rows")
except FileNotFoundError:
    print("Data file not found. Please ensure the path is correct.")

2026-05-26 20:53:22 [INFO] data_loader — Loading data from: c:\KAIM\MachineLearningRating_v3.txt
2026-05-26 20:53:37 [INFO] data_loader — Loaded 1,000,098 rows × 52 columns
2026-05-26 20:53:37 [INFO] data_loader — Starting preprocessing pipeline...
2026-05-26 20:53:41 [INFO] data_loader — Missing values after preprocessing:
NumberOfVehiclesInFleet    1000098
CrossBorder                 999400
log_CustomValueEstimate     779642
CustomValueEstimate         779642
Rebuilt                     641901
Converted                   641901
WrittenOff                  641901
LossRatio                   381634
NewVehicle                  153295
Bank                        145961
AccountType                  40232
Gender                        9536
MaritalStatus                 8259
make                           552
VehicleType                    552
mmcode                         552
Model                          552
cubiccapacity                  552
NumberOfDoors                  552
VehicleIn

Data loaded successfully: 1,000,098 rows


## 2. Hypothesis Testing

### H₀: No risk differences across provinces
We test Claim Frequency (Chi-Squared) and Claim Severity (T-test) across the top provinces.

In [3]:
# To keep it manageable, we test the top 2 provinces by policy count
top_provinces = df['Province'].value_counts().index[:2].tolist()
df_prov = df[df['Province'].isin(top_provinces)]

prov_results = test_province_risk(df_prov)
print(f"Completed {len(prov_results)} province tests.")

Completed 2 province tests.


### H₀: No risk differences between zip codes
We compare the two most populated zip codes.

In [4]:
zip_results = test_zipcode_risk(df)
print(f"Completed {len(zip_results)} zip code risk tests.")

Completed 2 zip code risk tests.


### H₀: No significant margin (profit) difference between zip codes

In [5]:
margin_results = test_margin_zipcode(df)
print(f"Completed {len(margin_results)} margin tests.")

Completed 1 margin tests.


### H₀: No significant risk difference between Women and Men

In [6]:
gender_results = test_gender_risk(df)
print(f"Completed {len(gender_results)} gender tests.")

Completed 2 gender tests.


## 3. Results Summary

In [7]:
all_tests = prov_results + zip_results + margin_results + gender_results
summary_df = build_results_table(all_tests)
summary_df

,Test,KPI,Group A,Group B,Mean/Freq A,Mean/Freq B,Statistic,p-value,Decision
0,Chi-Squared,Claim Frequency,Gauteng,Western Cape,0.0034,0.0022,56.0874,0.000000,Reject H₀
1,Welch's t-test,Claim Severity,Gauteng,Western Cape,22243.8784,28095.8499,-2.1685,0.030599,Reject H₀
2,Chi-Squared,Claim Frequency,2000,122,0.0036,0.0043,3.5971,0.057882,Fail to Reject H₀
3,Welch's t-test,Claim Severity,2000,122,19196.4137,18162.0259,0.3854,0.700208,Fail to Reject H₀
4,Welch's t-test,Margin,2000,122,-8.1119,-22.8598,1.1639,0.244462,Fail to Reject H₀
5,Chi-Squared,Claim Frequency,Female,Male,0.0021,0.0022,0.0037,0.951464,Fail to Reject H₀
6,Welch's t-test,Claim Severity,Female,Male,17874.7213,14858.5523,0.5790,0.568029,Fail to Reject H₀


## 4. Business Interpretations

Based on the results table above:

1. **Provinces**: If p < 0.05, we reject H₀. This implies geographic location is a significant driver of risk (either frequency or severity), and premiums should be adjusted by province.
2. **Zip Codes**: Significant differences in zip codes (p < 0.05) suggest that even within provinces, hyper-local risk factors exist (e.g., crime rates, traffic density).
3. **Margin**: If margin differences are significant, it indicates that current pricing is not effectively neutralizing the risk differences between locations, leading to "under-priced" or "over-priced" areas.
4. **Gender**: If gender risk is significant, it justifies gender-based segmentation (where legally permitted) to better reflect the risk profile of different demographic groups.